# Ejercicio 5: Espacio Vectorial
---
- **Nombre**: Michael Enríquez
- **Fecha**: 13 de mayo, 2026
---
## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [13]:
import os
import pandas as pd
import kagglehub

# Descargar dataset
dataset_path = kagglehub.dataset_download(
    "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects"
)

# Buscar el archivo CSV
csv_file = [f for f in os.listdir(dataset_path) if f.endswith(".csv")][0]
csv_path = os.path.join(dataset_path, csv_file)

# Cargar dataset
df = pd.read_csv(csv_path)

# Convertir a corpus de documentos
documents = df['text'].astype(str).tolist()

# IDs de documentos
doc_ids = [f"doc_{i}" for i in range(len(documents))]

print(f"\nDocumentos cargados: {len(documents)}")


Documentos cargados: 10859

Primer documento:
Anovo

Anovo (formerly A Novo) is a computer services company based in Beauvais, France. It was founded in 1987, went public in 1999, and is currently a member of the CAC Small.

It won in the category 'Service and Repair' of the Mobile News Awards four years in a row, from 2007 to 2010.

As of November 2017, they have a score of 1.6 out of 10 on the TrustPilot ratings site, with 86% of reviewers giving the company the lowest possible rating. 




In [15]:
# Crear Dataframe 
df = pd.DataFrame({'document': documents, 'doc_id': doc_ids})
df

,document,doc_id
0,Anovo\n\nAnovo (formerly A Novo) is a computer...,doc_0
1,Battery indicator\n\nA battery indicator (also...,doc_1
2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",doc_2
3,CAVNET\n\nCAVNET was a secure military forum w...,doc_3
4,CLidar\n\nThe CLidar is a scientific instrumen...,doc_4
...,...,...
10854,Soundcast\n\nSoundcast LLC is a privately fund...,doc_10854
10855,Spectrum analyzer\n\nA spectrum analyzer measu...,doc_10855
10856,Telepresence technology\n\nTelepresence techno...,doc_10856
10857,Trans-Pacific Profiler Network\n\nThe Trans-Pa...,doc_10857


In [18]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LabP5E004\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LabP5E004\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
# Limpiar datos:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def limpiar_texto(text):
    texto_lower = text.lower()
    texto_limpio = re.sub(r'[^a-zA-Z0-9\s]', '', texto_lower)
    tokens = word_tokenize(texto_limpio)

    tokens = [t for t in tokens if t not in stop_words]
    stems = [stemmer.stem(p) for p in tokens]

    return ' '.join(stems)

df['prep'] = df['document'].apply(limpiar_texto)
df

,document,doc_id,prep
0,Anovo\n\nAnovo (formerly A Novo) is a computer...,doc_0,anovo anovo formerli novo comput servic compan...
1,Battery indicator\n\nA battery indicator (also...,doc_1,batteri indic batteri indic also known batteri...
2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",doc_2,bob peas robert allen peas august 22 1940 june...
3,CAVNET\n\nCAVNET was a secure military forum w...,doc_3,cavnet cavnet secur militari forum becam oper ...
4,CLidar\n\nThe CLidar is a scientific instrumen...,doc_4,clidar clidar scientif instrument use measur p...
...,...,...,...
10854,Soundcast\n\nSoundcast LLC is a privately fund...,doc_10854,soundcast soundcast llc privat fund compani cr...
10855,Spectrum analyzer\n\nA spectrum analyzer measu...,doc_10855,spectrum analyz spectrum analyz measur magnitu...
10856,Telepresence technology\n\nTelepresence techno...,doc_10856,telepres technolog telepres technolog term use...
10857,Trans-Pacific Profiler Network\n\nThe Trans-Pa...,doc_10857,transpacif profil network transpacif profil ne...


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Crear vectorizador TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.8)

# Transformar documentos preprocesados
tfidf_matrix = vectorizer.fit_transform(df['prep'])

# Conjunto de 10 queries de prueba
queries = [
    "science and technology",
    "history of ancient civilizations",
    "literature and art",
    "mathematics and physics",
    "biological systems",
    "geography and culture",
    "political systems",
    "philosophy and ethics",
    "economics and trade",
    "nature and environment"
]

# Ejecutar búsquedas

print("RESULTADOS DE BÚSQUEDA TF-IDF\n")

for i, query in enumerate(queries, 1):
    # Preprocesar query
    query_prep = limpiar_texto(query)
    
    # Transformar query
    query_tfidf = vectorizer.transform([query_prep])
    
    # Calcular similitud
    df['dist'] = cosine_similarity(tfidf_matrix, query_tfidf).flatten()
    
    # Ordenar por similitud descendente
    results = df.sort_values(by='dist', ascending=False).head(3)
    
    # Mostrar resultados
    print(f"\nQuery {i}: '{query}'")
    print(f"   Preprocesado: '{query_prep}'\n")
    
    for idx, (_, row) in enumerate(results.iterrows(), 1):
        if row['dist'] > 0:
            print(f"  {idx}. [{row['doc_id']}] Similitud: {row['dist']:.4f}")
            print(f"     {row['document'][:100]}...")

df

RESULTADOS DE BÚSQUEDA TF-IDF


Query 1: 'science and technology'
   Preprocesado: 'scienc technolog'

  1. [doc_618] Similitud: 0.6559
     Ministry of Science and Technology (Bangladesh)

The Ministry of Science and Technology (; "BijÃ±Än...
  2. [doc_3842] Similitud: 0.6515
     Ministry of Science and Technology (China)

The Ministry of Science and Technology (MOST) of the Peo...
  3. [doc_1317] Similitud: 0.6188
     Ministry of Science and Technology (Taiwan)

The Ministry of Science and Technology (MOST; ) is the ...

Query 2: 'history of ancient civilizations'
   Preprocesado: 'histori ancient civil'

  1. [doc_2186] Similitud: 0.3580
     Civil engineer

A civil engineer is a person who practices civil engineering â€“ the application of ...
  2. [doc_2332] Similitud: 0.3335
     Civil engineering database

The Civil Engineering Database (CEDB) was created in 1994, and is mainta...
  3. [doc_1884] Similitud: 0.2999
     Advanced Technician in Aviation non civil servant

In Fra

,document,doc_id,prep,dist
0,Anovo\n\nAnovo (formerly A Novo) is a computer...,doc_0,anovo anovo formerli novo comput servic compan...,0.000000
1,Battery indicator\n\nA battery indicator (also...,doc_1,batteri indic batteri indic also known batteri...,0.000000
2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",doc_2,bob peas robert allen peas august 22 1940 june...,0.000000
3,CAVNET\n\nCAVNET was a secure military forum w...,doc_3,cavnet cavnet secur militari forum becam oper ...,0.000000
4,CLidar\n\nThe CLidar is a scientific instrumen...,doc_4,clidar clidar scientif instrument use measur p...,0.000000
...,...,...,...,...
10854,Soundcast\n\nSoundcast LLC is a privately fund...,doc_10854,soundcast soundcast llc privat fund compani cr...,0.000000
10855,Spectrum analyzer\n\nA spectrum analyzer measu...,doc_10855,spectrum analyz spectrum analyz measur magnitu...,0.006694
10856,Telepresence technology\n\nTelepresence techno...,doc_10856,telepres technolog telepres technolog term use...,0.078498
10857,Trans-Pacific Profiler Network\n\nThe Trans-Pa...,doc_10857,transpacif profil network transpacif profil ne...,0.000000


## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [33]:
from rank_bm25 import BM25Okapi

# Tokenizar documentos preprocesados (ya están limpios)
tokenized_docs = [doc.split() for doc in df['prep']]

# Crear modelo BM25
bm25 = BM25Okapi(tokenized_docs)

print(f"Corpus BM25 creado: {len(tokenized_docs)} documentos")

# Ejecutar búsquedas
print("\nRESULTADOS DE BÚSQUEDA BM25")

for i, query in enumerate(queries, 1):
    # Preprocesar query
    query_prep = limpiar_texto(query)
    query_tokens = query_prep.split()
    
    # Calcular scores BM25
    scores = bm25.get_scores(query_tokens)
    df['score'] = scores
    
    # Ordenar por score descendente
    results = df.sort_values(by='score', ascending=False).head(3)
    
    # Mostrar resultados
    print(f"\nQuery {i}: '{query}'")
    print(f"   Preprocesado: '{query_prep}'")
    print("-" * 80)
    
    for idx, (_, row) in enumerate(results.iterrows(), 1):
        if row['dist'] > 0:
            print(f"  {idx}. [{row['doc_id']}] Score BM25: {row['dist']:.4f}")
            print(f"     {row['document'][:100]}...")

df

Corpus BM25 creado: 10859 documentos

RESULTADOS DE BÚSQUEDA BM25

Query 1: 'science and technology'
   Preprocesado: 'scienc technolog'
--------------------------------------------------------------------------------

Query 2: 'history of ancient civilizations'
   Preprocesado: 'histori ancient civil'
--------------------------------------------------------------------------------
  3. [doc_3359] Score BM25: 0.0294
     Ancient Egyptian technology

Ancient Egyptian technology describes devices and technologies invented...

Query 3: 'literature and art'
   Preprocesado: 'literatur art'
--------------------------------------------------------------------------------

Query 4: 'mathematics and physics'
   Preprocesado: 'mathemat physic'
--------------------------------------------------------------------------------
  2. [doc_7027] Score BM25: 0.0187
     Louis Kauffman

Louis Hirsch Kauffman (born February 3, 1945) is an American mathematician, topologi...

Query 5: 'biological systems'

,document,doc_id,prep,dist,score
0,Anovo\n\nAnovo (formerly A Novo) is a computer...,doc_0,anovo anovo formerli novo comput servic compan...,0.000000,0.000000
1,Battery indicator\n\nA battery indicator (also...,doc_1,batteri indic batteri indic also known batteri...,0.000000,0.000000
2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",doc_2,bob peas robert allen peas august 22 1940 june...,0.000000,0.000000
3,CAVNET\n\nCAVNET was a secure military forum w...,doc_3,cavnet cavnet secur militari forum becam oper ...,0.000000,0.000000
4,CLidar\n\nThe CLidar is a scientific instrumen...,doc_4,clidar clidar scientif instrument use measur p...,0.000000,0.000000
...,...,...,...,...,...
10854,Soundcast\n\nSoundcast LLC is a privately fund...,doc_10854,soundcast soundcast llc privat fund compani cr...,0.000000,0.000000
10855,Spectrum analyzer\n\nA spectrum analyzer measu...,doc_10855,spectrum analyz spectrum analyz measur magnitu...,0.006694,1.656911
10856,Telepresence technology\n\nTelepresence techno...,doc_10856,telepres technolog telepres technolog term use...,0.078498,3.143713
10857,Trans-Pacific Profiler Network\n\nThe Trans-Pa...,doc_10857,transpacif profil network transpacif profil ne...,0.000000,0.000000


## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [36]:
import pandas as pd

# Almacenar resultados de ambos modelos
results_comparison = []

print("COMPARACIÓN TF-IDF vs BM25")

for i, query in enumerate(queries, 1):
    # Preprocesar query
    query_prep = limpiar_texto(query)
    query_tokens = query_prep.split()
    
    # ========== TF-IDF ==========
    query_tfidf = vectorizer.transform([query_prep])
    df['tfidf_score'] = cosine_similarity(tfidf_matrix, query_tfidf).flatten()
    tfidf_results = df.sort_values(by='tfidf_score', ascending=False).head(5)
    
    # ========== BM25 ==========
    scores = bm25.get_scores(query_tokens)
    df['bm25_score'] = scores
    bm25_results = df.sort_values(by='bm25_score', ascending=False).head(5)
    
    # Extraer doc_ids para comparación
    tfidf_docs = tfidf_results['doc_id'].values
    bm25_docs = bm25_results['doc_id'].values
    
    # Calcular coincidencias
    coincidencias = set(tfidf_docs) & set(bm25_docs)
    
    # Mostrar resultados
    print(f"\nQuery {i}: '{query}'")
    print(f"   Preprocesado: '{query_prep}'\n")
    
    # TF-IDF Top 5
    print("\n   TF-IDF Top 5:")
    for rank, (_, row) in enumerate(tfidf_results.iterrows(), 1):
        if row['tfidf_score'] > 0:
            print(f"    {rank}. {row['doc_id']:<10} Score: {row['tfidf_score']:.4f}")
    
    # BM25 Top 5
    print("\n   BM25 Top 5:")
    for rank, (_, row) in enumerate(bm25_results.iterrows(), 1):
        if row['bm25_score'] > 0:
            print(f"    {rank}. {row['doc_id']:<10} Score: {row['bm25_score']:.4f}")
    
    # Análisis de coincidencias
    print(f"\n\tAnálisis: {len(coincidencias)}/5 documentos coinciden")
    if coincidencias:
        print(f"\tDocumentos en común: {', '.join(sorted(coincidencias))}")
    
    # Guardar resultados para análisis posterior
    results_comparison.append({
        'query': query,
        'tfidf': tfidf_docs[0] if len(tfidf_docs) > 0 else None,
        'bm25': bm25_docs[0] if len(bm25_docs) > 0 else None,
        'tfidf_docs': list(tfidf_docs),
        'bm25_docs': list(bm25_docs),
        'coincidencias': len(coincidencias)
    })

# Crear DataFrame de comparación
comp_df = pd.DataFrame(results_comparison)

# Resumen global

print("\nRESUMEN GLOBAL")
print(f"\nTotal de queries: {len(queries)}")
print(f"Top 1 coincidencias: {(comp_df['tfidf'] == comp_df['bm25']).sum()}/10")
print(f"Promedio de docs coincidentes en Top 5: {comp_df['coincidencias'].mean():.2f}/5")
print(f"\nDistribución de coincidencias:")
print(comp_df['coincidencias'].value_counts().sort_index(ascending=False))

# Mostrar tabla de comparación
print("\nTABLA RESUMEN")
display(comp_df[['query', 'tfidf', 'bm25', 'coincidencias']])


COMPARACIÓN TF-IDF vs BM25

Query 1: 'science and technology'
   Preprocesado: 'scienc technolog'


   TF-IDF Top 5:
    1. doc_618    Score: 0.6559
    2. doc_3842   Score: 0.6515
    3. doc_1317   Score: 0.6188
    4. doc_6377   Score: 0.5869
    5. doc_8563   Score: 0.5470

   BM25 Top 5:
    1. doc_5501   Score: 6.0734
    2. doc_1317   Score: 5.9769
    3. doc_6377   Score: 5.9754
    4. doc_5493   Score: 5.9191
    5. doc_7921   Score: 5.8937

	Análisis: 2/5 documentos coinciden
	Documentos en común: doc_1317, doc_6377

Query 2: 'history of ancient civilizations'
   Preprocesado: 'histori ancient civil'


   TF-IDF Top 5:
    1. doc_2186   Score: 0.3580
    2. doc_2332   Score: 0.3335
    3. doc_1884   Score: 0.2999
    4. doc_8828   Score: 0.2986
    5. doc_340    Score: 0.2859

   BM25 Top 5:
    1. doc_8776   Score: 15.2221
    2. doc_4265   Score: 14.2552
    3. doc_3359   Score: 14.0373
    4. doc_8901   Score: 12.9844
    5. doc_2854   Score: 12.6198

	Análisis: 0/5 documen

,query,tfidf,bm25,coincidencias
0,science and technology,doc_618,doc_5501,2
1,history of ancient civilizations,doc_2186,doc_8776,0
2,literature and art,doc_5475,doc_9207,0
3,mathematics and physics,doc_1734,doc_3757,2
4,biological systems,doc_5592,doc_5592,2
5,geography and culture,doc_6032,doc_10074,0
6,political systems,doc_4823,doc_4823,2
7,philosophy and ethics,doc_9852,doc_6579,3
8,economics and trade,doc_2570,doc_4910,0
9,nature and environment,doc_8911,doc_1349,0
